# Manipulation Pipeline Demo with ContactGraspNet

This notebook demonstrates the complete manipulation pipeline including:
- Object detection (Detic)
- Semantic segmentation (SAM/FastSAM)
- Point cloud processing
- 6-DoF grasp generation (ContactGraspNet)
- 3D visualization

---

## 1. Setup and Imports

In [1]:
import os
import sys
import cv2
import numpy as np
import time
import matplotlib

# Configure matplotlib backend
try:
    matplotlib.use("TkAgg")
except:
    try:
        matplotlib.use("Qt5Agg")
    except:
        matplotlib.use("Agg")

import matplotlib.pyplot as plt
import open3d as o3d
from typing import Dict, List

# Add project root to path
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

# Import DIMOS modules
from dimos.perception.pointcloud.utils import visualize_clustered_point_clouds, visualize_voxel_grid
from dimos.perception.manip_aio_processer import ManipulationProcessor
from dimos.perception.pointcloud.utils import load_camera_matrix_from_yaml, visualize_pcd
from dimos.utils.logging_config import setup_logger

# Import ContactGraspNet visualization
from dimos.models.manipulation.contact_graspnet_pytorch.contact_graspnet_pytorch.visualization_utils_o3d import (
    visualize_grasps,
)

logger = setup_logger("manipulation_pipeline_demo")
print("✅ All imports successful!")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
✅ All imports successful!


## 2. Configuration

In [2]:
# Configuration parameters
CONFIG = {
    "data_dir": "/home/alex-lin/dimos/assets/rgbd_data",
    "enable_grasp_generation": True,
    "enable_segmentation": True,
    "segmentation_model": "FastSAM-x.pt",  # or "sam2_b.pt"
    "min_confidence": 0.6,
    "max_objects": 20,
    "show_3d_visualizations": True,
    "save_results": True,
}

print(f"Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Configuration:
  data_dir: /home/alex-lin/dimos/assets/rgbd_data
  enable_grasp_generation: True
  enable_segmentation: True
  segmentation_model: FastSAM-x.pt
  min_confidence: 0.6
  max_objects: 20
  show_3d_visualizations: True
  save_results: True


## 3. Data Loading Functions

In [3]:
def load_first_frame(data_dir: str):
    """Load first RGB-D frame and camera intrinsics."""
    # Load images
    color_img = cv2.imread(os.path.join(data_dir, "color", "00000.png"))
    color_img = cv2.cvtColor(color_img, cv2.COLOR_BGR2RGB)

    depth_img = cv2.imread(os.path.join(data_dir, "depth", "00000.png"), cv2.IMREAD_ANYDEPTH)
    if depth_img.dtype == np.uint16:
        depth_img = depth_img.astype(np.float32) / 1000.0

    # Load intrinsics
    camera_matrix = load_camera_matrix_from_yaml(os.path.join(data_dir, "color_camera_info.yaml"))
    intrinsics = [
        camera_matrix[0, 0],  # fx
        camera_matrix[1, 1],  # fy
        camera_matrix[0, 2],  # cx
        camera_matrix[1, 2],  # cy
    ]

    return color_img, depth_img, intrinsics


def create_point_cloud(color_img, depth_img, intrinsics):
    """Create Open3D point cloud for reference."""
    fx, fy, cx, cy = intrinsics
    height, width = depth_img.shape

    o3d_intrinsics = o3d.camera.PinholeCameraIntrinsic(width, height, fx, fy, cx, cy)
    color_o3d = o3d.geometry.Image(color_img)
    depth_o3d = o3d.geometry.Image((depth_img * 1000).astype(np.uint16))

    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color_o3d, depth_o3d, depth_scale=1000.0, convert_rgb_to_intensity=False
    )

    return o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, o3d_intrinsics)


print("✅ Data loading functions defined!")

✅ Data loading functions defined!


## 4. Load RGB-D Data

In [4]:
# Load data
color_img, depth_img, intrinsics = load_first_frame(CONFIG["data_dir"])
logger.info(f"Loaded images: color {color_img.shape}, depth {depth_img.shape}")

# Display input images
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(color_img)
axes[0].set_title("RGB Image")
axes[0].axis("off")

# Colorize depth for visualization
depth_colorized = cv2.applyColorMap(
    cv2.convertScaleAbs(depth_img, alpha=255.0 / depth_img.max()), cv2.COLORMAP_JET
)
depth_colorized = cv2.cvtColor(depth_colorized, cv2.COLOR_BGR2RGB)
axes[1].imshow(depth_colorized)
axes[1].set_title("Depth Image")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print(
    f"Camera intrinsics: fx={intrinsics[0]:.1f}, fy={intrinsics[1]:.1f}, cx={intrinsics[2]:.1f}, cy={intrinsics[3]:.1f}"
)

2025-06-25 15:04:05,846 - manipulation_pipeline_demo - INFO - Loaded images: color (720, 1280, 3), depth (720, 1280)


Camera intrinsics: fx=749.3, fy=748.6, cx=639.4, cy=357.2


## 5. Initialize Manipulation Processor

In [5]:
# Create processor with ContactGraspNet enabled
processor = ManipulationProcessor(
    camera_intrinsics=intrinsics,
    min_confidence=CONFIG["min_confidence"],
    max_objects=CONFIG["max_objects"],
    enable_grasp_generation=CONFIG["enable_grasp_generation"],
    enable_segmentation=CONFIG["enable_segmentation"],
    segmentation_model=CONFIG["segmentation_model"],
)

print("✅ ManipulationProcessor initialized successfully!")

/home/alex-lin/miniconda3/envs/contact-graspnet/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/alex-lin/miniconda3/envs/contact-graspnet/lib/python3.10/site-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/alex-lin/miniconda3/envs/contact-graspnet/lib/python3.10/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/alex-lin/miniconda3/envs/contact-graspnet/lib/python3.10/site-packages/t

Resetting zs_weight /home/alex-lin/dimos/dimos/perception/detection2d/../../models/Detic/datasets/metadata/lvis_v1_clip_a+cname.npy
model func:  <module 'contact_graspnet_pytorch.contact_graspnet' from '/home/alex-lin/dimos/dimos/models/manipulation/contact_graspnet_pytorch/contact_graspnet_pytorch/contact_graspnet.py'>
✅ ManipulationProcessor initialized successfully!


## 6. Run Processing Pipeline

In [6]:
# Process single frame
print("🔄 Processing frame through pipeline...")
start_time = time.time()

results = processor.process_frame(color_img, depth_img)

processing_time = time.time() - start_time
print(f"✅ Processing completed in {processing_time:.3f}s")

🔄 Processing frame through pipeline...
DBSCAN clustering found 13 clusters from 26536 points
Created voxel grid with 2074 voxels (voxel_size=0.02)
using local regions
Extracted Region Cube Size:  0.3148665130138397
Extracted Region Cube Size:  0.4740000367164612
Extracted Region Cube Size:  0.2676139771938324
Extracted Region Cube Size:  0.2
Extracted Region Cube Size:  0.4960000514984131
Extracted Region Cube Size:  0.30400002002716064
Extracted Region Cube Size:  0.38946154713630676
Extracted Region Cube Size:  0.2087651789188385
Extracted Region Cube Size:  0.2
Extracted Region Cube Size:  0.2
Extracted Region Cube Size:  0.24777960777282715
Extracted Region Cube Size:  0.2
Extracted Region Cube Size:  0.2502080202102661
Extracted Region Cube Size:  0.2
Extracted Region Cube Size:  0.3400000333786011
Extracted Region Cube Size:  0.22946105897426605
Extracted Region Cube Size:  0.2
Extracted Region Cube Size:  0.5360000133514404
Generated 18 grasps for object 2
Generated 44 grasps fo

2025-06-25 15:04:30,107 - dimos.perception.grasp_generation - INFO - Generated 296 grasps across 18 objects in 14.69s


Generated 23 grasps for object 37
✅ Processing completed in 18.517s


## 7. Results Summary

In [7]:
# Print comprehensive results summary
print(f"\n📊 PROCESSING RESULTS SUMMARY")
print(f"" + "=" * 50)
print(f"Available results: {list(results.keys())}")
print(f"Total processing time: {results.get('processing_time', 0):.3f}s")

# Show timing breakdown
if "timing_breakdown" in results:
    breakdown = results["timing_breakdown"]
    print(f"\n⏱️  Timing breakdown:")
    print(f"   Detection: {breakdown.get('detection', 0):.3f}s")
    print(f"   Segmentation: {breakdown.get('segmentation', 0):.3f}s")
    print(f"   Point cloud: {breakdown.get('pointcloud', 0):.3f}s")
    print(f"   Misc extraction: {breakdown.get('misc_extraction', 0):.3f}s")

# Object counts
detected_count = len(results.get("detected_objects", []))
all_count = len(results.get("all_objects", []))
print(f"\n🎯 Object Detection:")
print(f"   Detection objects: {detected_count}")
print(f"   All objects processed: {all_count}")

# Misc clusters info
if "misc_clusters" in results and results["misc_clusters"]:
    cluster_count = len(results["misc_clusters"])
    total_misc_points = sum(len(np.asarray(cluster.points)) for cluster in results["misc_clusters"])
    print(f"\n🧩 Background Analysis:")
    print(f"   Misc clusters: {cluster_count} clusters with {total_misc_points:,} total points")
else:
    print(f"\n🧩 Background Analysis: No clusters found")

# ContactGraspNet grasp summary
if "grasps" in results and results["grasps"]:
    grasp_data = results["grasps"]
    if isinstance(grasp_data, dict):
        pred_grasps = grasp_data.get("pred_grasps_cam", {})
        scores = grasp_data.get("scores", {})

        total_grasps = 0
        best_score = 0

        for obj_id, obj_grasps in pred_grasps.items():
            num_grasps = len(obj_grasps) if hasattr(obj_grasps, "__len__") else 0
            total_grasps += num_grasps

            if obj_id in scores and len(scores[obj_id]) > 0:
                obj_best_score = max(scores[obj_id])
                if obj_best_score > best_score:
                    best_score = obj_best_score

        print(f"\n🤖 ContactGraspNet Results:")
        print(f"   Total grasps: {total_grasps}")
        print(f"   Best score: {best_score:.3f}")
        print(f"   Objects with grasps: {len(pred_grasps)}")
    else:
        print(f"\n🤖 ContactGraspNet Results: Invalid format")
else:
    print(f"\n🤖 ContactGraspNet Results: No grasps generated")

print("\n" + "=" * 50)


📊 PROCESSING RESULTS SUMMARY
Available results: ['detection2d_objects', 'detection_viz', 'segmentation2d_objects', 'segmentation_viz', 'detected_objects', 'all_objects', 'full_pointcloud', 'misc_clusters', 'misc_voxel_grid', 'pointcloud_viz', 'detected_pointcloud_viz', 'misc_pointcloud_viz', 'grasps', 'processing_time', 'timing_breakdown']
Total processing time: 18.517s

⏱️  Timing breakdown:
   Detection: 0.529s
   Segmentation: 0.720s
   Point cloud: 1.837s
   Misc extraction: 0.385s

🎯 Object Detection:
   Detection objects: 13
   All objects processed: 18

🧩 Background Analysis:
   Misc clusters: 13 clusters with 25,628 total points

🤖 ContactGraspNet Results:
   Total grasps: 296
   Best score: 0.798
   Objects with grasps: 18



## 8. 2D Visualization Results

In [8]:
# Collect available visualizations
viz_configs = []

if "detection_viz" in results and results["detection_viz"] is not None:
    viz_configs.append(("detection_viz", "Object Detection"))

if "segmentation_viz" in results and results["segmentation_viz"] is not None:
    viz_configs.append(("segmentation_viz", "Semantic Segmentation"))

if "pointcloud_viz" in results and results["pointcloud_viz"] is not None:
    viz_configs.append(("pointcloud_viz", "All Objects Point Cloud"))

if "detected_pointcloud_viz" in results and results["detected_pointcloud_viz"] is not None:
    viz_configs.append(("detected_pointcloud_viz", "Detection Objects Point Cloud"))

if "misc_pointcloud_viz" in results and results["misc_pointcloud_viz"] is not None:
    viz_configs.append(("misc_pointcloud_viz", "Misc/Background Points"))

# Create visualization grid
if viz_configs:
    num_plots = len(viz_configs)

    if num_plots <= 3:
        fig, axes = plt.subplots(1, num_plots, figsize=(6 * num_plots, 5))
    else:
        rows = 2
        cols = (num_plots + 1) // 2
        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))

    # Ensure axes is always iterable
    if num_plots == 1:
        axes = [axes]
    elif num_plots > 2:
        axes = axes.flatten()

    # Plot each result
    for i, (key, title) in enumerate(viz_configs):
        axes[i].imshow(results[key])
        axes[i].set_title(title, fontsize=12, fontweight="bold")
        axes[i].axis("off")

    # Hide unused subplots
    if num_plots > 3:
        for i in range(num_plots, len(axes)):
            axes[i].axis("off")

    plt.tight_layout()

    if CONFIG["save_results"]:
        output_path = "manipulation_results.png"
        plt.savefig(output_path, dpi=150, bbox_inches="tight")
        print(f"📸 Results saved to: {output_path}")

    plt.show()
else:
    print("⚠️  No 2D visualization results to display")

📸 Results saved to: manipulation_results.png


## 9. 3D ContactGraspNet Visualization

In [9]:
# 3D ContactGraspNet visualization
if (
    CONFIG["show_3d_visualizations"]
    and "grasps" in results
    and results["grasps"]
    and "full_pointcloud" in results
):
    grasp_data = results["grasps"]
    full_pcd = results["full_pointcloud"]

    if isinstance(grasp_data, dict) and full_pcd is not None:
        try:
            # Extract ContactGraspNet data
            pred_grasps_cam = grasp_data.get("pred_grasps_cam", {})
            scores = grasp_data.get("scores", {})
            contact_pts = grasp_data.get("contact_pts", {})
            gripper_openings = grasp_data.get("gripper_openings", {})

            # Check if we have valid grasp data
            total_grasps = (
                sum(len(grasps) for grasps in pred_grasps_cam.values()) if pred_grasps_cam else 0
            )

            if total_grasps > 0:
                print(
                    f"🎯 Launching 3D visualization with {total_grasps} ContactGraspNet grasps..."
                )
                print("📌 Note: Close the 3D window to continue with the notebook")

                # Use ContactGraspNet's native visualization - pass dictionaries directly
                visualize_grasps(
                    full_pcd,
                    pred_grasps_cam,  # Pass dictionary directly
                    scores,  # Pass dictionary directly
                    gripper_openings=gripper_openings,
                )

                print("✅ 3D grasp visualization completed!")
            else:
                print("⚠️  No valid grasps to visualize in 3D")

        except Exception as e:
            print(f"❌ Error in ContactGraspNet 3D visualization: {e}")
            print("   Skipping 3D grasp visualization")
else:
    if not CONFIG["show_3d_visualizations"]:
        print("⏭️  3D visualizations disabled in config")
    else:
        print("⚠️  ContactGraspNet grasp generation disabled or no results")

🎯 Launching 3D visualization with 296 ContactGraspNet grasps...
📌 Note: Close the 3D window to continue with the notebook
Visualizing...
✅ 3D grasp visualization completed!


## 10. Additional 3D Visualizations

### 10.1 Full Scene Point Cloud

In [10]:
if (
    CONFIG["show_3d_visualizations"]
    and "full_pointcloud" in results
    and results["full_pointcloud"] is not None
):
    full_pcd = results["full_pointcloud"]
    num_points = len(np.asarray(full_pcd.points))
    print(f"🌍 Launching full scene point cloud visualization ({num_points:,} points)...")
    print("📌 Note: Close the 3D window to continue")

    try:
        visualize_pcd(
            full_pcd,
            window_name="Full Scene Point Cloud",
            point_size=2.0,
            show_coordinate_frame=True,
        )
        print("✅ Full point cloud visualization completed!")
    except (KeyboardInterrupt, EOFError):
        print("⏭️  Full point cloud visualization skipped")
else:
    print("⚠️  No full point cloud available for visualization")

🌍 Launching full scene point cloud visualization (526,100 points)...
📌 Note: Close the 3D window to continue
Visualizing point cloud with 526100 points
✅ Full point cloud visualization completed!


### 10.2 Background/Misc Clusters

In [11]:
if CONFIG["show_3d_visualizations"] and "misc_clusters" in results and results["misc_clusters"]:
    misc_clusters = results["misc_clusters"]
    cluster_count = len(misc_clusters)
    total_misc_points = sum(len(np.asarray(cluster.points)) for cluster in misc_clusters)

    print(
        f"🧩 Launching misc/background clusters visualization ({cluster_count} clusters, {total_misc_points:,} points)..."
    )
    print("📌 Note: Close the 3D window to continue")

    try:
        visualize_clustered_point_clouds(
            misc_clusters,
            window_name="Misc/Background Clusters (DBSCAN)",
            point_size=3.0,
            show_coordinate_frame=True,
        )
        print("✅ Misc clusters visualization completed!")
    except (KeyboardInterrupt, EOFError):
        print("⏭️  Misc clusters visualization skipped")
else:
    print("⚠️  No misc clusters available for visualization")

🧩 Launching misc/background clusters visualization (13 clusters, 25,628 points)...
📌 Note: Close the 3D window to continue
Visualizing 13 clusters with 25628 total points
✅ Misc clusters visualization completed!


### 10.3 Voxel Grid Visualization

In [12]:
if (
    CONFIG["show_3d_visualizations"]
    and "misc_voxel_grid" in results
    and results["misc_voxel_grid"] is not None
):
    misc_voxel_grid = results["misc_voxel_grid"]
    voxel_count = len(misc_voxel_grid.get_voxels())

    print(f"📦 Launching voxel grid visualization ({voxel_count} voxels)...")
    print("📌 Note: Close the 3D window to continue")

    try:
        visualize_voxel_grid(
            misc_voxel_grid,
            window_name="Misc/Background Voxel Grid",
            show_coordinate_frame=True,
        )
        print("✅ Voxel grid visualization completed!")
    except (KeyboardInterrupt, EOFError):
        print("⏭️  Voxel grid visualization skipped")
    except Exception as e:
        print(f"❌ Error in voxel grid visualization: {e}")
else:
    print("⚠️  No voxel grid available for visualization")

📦 Launching voxel grid visualization (2074 voxels)...
📌 Note: Close the 3D window to continue
Visualizing voxel grid with 2074 voxels
✅ Voxel grid visualization completed!


## 11. Cleanup

In [13]:
# Clean up resources
processor.cleanup()
print("✅ Pipeline cleanup completed!")
print("\n🎉 Manipulation pipeline demo finished successfully!")

2025-06-25 15:05:01,624 - dimos.perception.grasp_generation - INFO - ContactGraspNet grasp generator cleaned up
2025-06-25 15:05:01,626 - dimos.perception.manip_aio_processor - INFO - ManipulationProcessor cleaned up


✅ Pipeline cleanup completed!

🎉 Manipulation pipeline demo finished successfully!


---

## Summary

This notebook demonstrated the complete DIMOS manipulation pipeline:

1. **Object Detection**: Using Detic for 2D object detection
2. **Semantic Segmentation**: Using SAM/FastSAM for detailed segmentation
3. **Point Cloud Processing**: Converting RGB-D to 3D point clouds with filtering
4. **Background Analysis**: DBSCAN clustering of miscellaneous points
5. **Grasp Generation**: ContactGraspNet for 6-DoF grasp pose estimation
6. **Visualization**: Comprehensive 2D and 3D visualizations

The pipeline is designed for real-time robotic manipulation tasks and provides rich visual feedback for debugging and analysis.

### Key Features:
- ✅ Modular design with clean interfaces
- ✅ Real-time performance optimizations
- ✅ Comprehensive error handling
- ✅ Rich visualization capabilities
- ✅ ContactGraspNet integration for state-of-the-art grasp generation

### Next Steps:
- Integrate with robotic control systems
- Add grasp execution and feedback
- Implement multi-frame tracking
- Add custom object vocabularies
